# Sprint 6 — 01 Model Packaging

**Objective:** validate `rag_model.py` — the deployable MLflow **PythonModel (PyFunc)** that wraps the Sprint 4 RAG pipeline — and prove it can be packaged, saved, and loaded **independently of the development notebooks**.

## Architecture decision

* **MLflow PyFunc** (`Civ4RagModel`) — the Databricks-recommended packaging for RAG applications. Rejected alternatives: agent/LangChain flavors (heavier, no benefit — the validated pipeline is plain PyFunc-compatible code) and Databricks Apps (a different product, not Model Serving).
* **Zero business-logic duplication:** every pipeline stage calls `rag_utils` (Sprint 4), shipped alongside `rag_model.py` via MLflow `code_paths`. The packaged model = orchestrator over two external Databricks services (Vector Search + Foundation Model APIs) — the PyFunc abstraction models this correctly; it does **not** pretend the RAG pipeline is a conventional predictive model.
* **Pinned config:** Sprint 5 **Config B** (`top_k=10`, prompt v1, temperature 0.1) — the best validated configuration (`civ4_evaluation_summary`, eval dataset `v1-1fd1d9e2`).

## Inputs / outputs

* **Inputs:** `04_generation/rag_utils.py` (read-only), Vector Search endpoint/index, `databricks-qwen3-next-80b-a3b-instruct` LLM endpoint, questions from `workspace.gcivilization.civ4_eval_questions`.
* **Outputs:** a validated trial model artifact (saved + re-loaded from disk) proving packaging works. **MLflow logging + UC registration happen in `02_model_registration`** — this notebook does not write to MLflow.

## Checks

1. Environment prerequisites (VS endpoint/index, LLM endpoint ready)
2. Model signature contract (input `question` → output `answer` + `references`)
3. Structural validation (input normalization, empty/invalid questions, `load_context`)
4. Save → load → predict round trip from a standalone artifact (with `code_paths`)
5. Retrieval parity with the Sprint 4 implementation (deterministic document-level check)
6. Out-of-domain behavior preserved

In [0]:
# Setup — import the shared Sprint 4 module and the new deployable model module.
# Both live in the repo and are shipped together as MLflow code_paths.
import sys
for _dir in (
    "/Workspace/Repos/endoerner@gmail.com/civ4-rag/04_generation",
    "/Workspace/Repos/endoerner@gmail.com/civ4-rag/06_model_serving",
):
    if _dir not in sys.path:
        sys.path.insert(0, _dir)

import json
import pandas as pd
import rag_model
import rag_utils

print("Pinned deployment configuration (Sprint 5, Config B):")
print(json.dumps(rag_model.PINNED_CONFIG, indent=2))
assert rag_model.PINNED_CONFIG["top_k"] == 10
assert rag_model.PINNED_CONFIG["prompt_version"] == "v1"
assert rag_model.PINNED_CONFIG["temperature"] == rag_utils.LLM_TEMPERATURE == 0.1
print("PASS  Config B pinned and consistent with rag_utils defaults")

In [0]:
# Check 1 — environment prerequisites the deployed model will call at inference time.
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Vector Search endpoint + index
vs_endpoint = w.vector_search_endpoints.get_endpoint(rag_utils.__dict__.get("VS_ENDPOINT", "civ4_vs_endpoint"))
idx = w.vector_search_indexes.get_index(index_name=rag_utils.INDEX_NAME)
vs_state = getattr(vs_endpoint.endpoint_status.state, "value", vs_endpoint.endpoint_status.state)
assert vs_state == "ONLINE", f"VS endpoint state: {vs_state}"
assert idx.status.ready, "VS index is not ready"
assert idx.status.indexed_row_count == 604, f"indexed rows: {idx.status.indexed_row_count}"
print(f"PASS  Vector Search: endpoint={vs_endpoint.name} ({vs_endpoint.endpoint_status.state}), "
      f"index={idx.name} ready with {idx.status.indexed_row_count} rows")

# LLM + embedding endpoints (Foundation Model APIs)
for ep_name in (rag_utils.LLM_ENDPOINT, rag_utils.EMBEDDING_MODEL):
    ep = w.serving_endpoints.get(ep_name)
    ready = getattr(ep.state.ready, "value", ep.state.ready)
    assert ready == "READY", f"{ep_name} state: {ready}"
print(f"PASS  LLM endpoint {rag_utils.LLM_ENDPOINT} READY")
print(f"PASS  Embedding endpoint {rag_utils.EMBEDDING_MODEL} READY")

In [0]:
# Check 2 — model signature contract.
# Input:  one 'question' string column (dataframe_records at the REST boundary).
# Output: 'answer' string + 'references' array of {document_id, title, source_url, author, category}.
from mlflow.models import ModelSignature

sig = ModelSignature(rag_model.INPUT_SCHEMA, rag_model.OUTPUT_SCHEMA)
assert "question" in str(rag_model.INPUT_SCHEMA)
assert "answer" in str(rag_model.OUTPUT_SCHEMA) and "references" in str(rag_model.OUTPUT_SCHEMA)
for field in ("document_id", "title", "source_url"):
    assert field in str(rag_model.OUTPUT_SCHEMA), f"references missing {field}"
print("INPUT :", str(rag_model.INPUT_SCHEMA))
print("OUTPUT:", str(rag_model.OUTPUT_SCHEMA))
print("PASS  signature contract complete")

In [0]:
# Check 3 — structural validation (no service calls): input normalization,
# empty/invalid questions, and the load_context hook.
model = rag_model.Civ4RagModel()

# Input normalization: str / dict / DataFrame all accepted
assert model._normalize_questions("What units counter axes?") == ["What units counter axes?"]
assert model._normalize_questions({"question": "abc"}) == ["abc"]
assert model._normalize_questions(pd.DataFrame({"question": ["a", "b"]})) == ["a", "b"]

# Malformed input raises an informative error (never a silent crash)
try:
    model._normalize_questions({"wrong": 1})
    raise AssertionError("malformed input should fail")
except ValueError as e:
    assert "question" in str(e)

# Empty / invalid questions return a structured response with empty references
for bad in ("", "   ", None, 123):
    r = model._answer_one(bad)
    assert r["references"] == [] and isinstance(r["answer"], str) and r["answer"].strip()

# load_context validation hook passes with the shipped rag_utils
class _Ctx:
    pass
model.load_context(_Ctx())
print("PASS  structural validation (normalization, empty/invalid input, load_context)")

In [0]:
# Check 4 — save -> load -> predict round trip from a STANDALONE artifact.
# This proves the component is packaged independently of the development
# notebook: rag_model.py + rag_utils.py are copied INTO the artifact via
# code_paths, then the artifact is loaded from disk and answers a real question.
import tempfile
import mlflow

# Representative question from the Sprint 5 evaluation dataset
q_row = spark.sql("""
    SELECT question, expected_document_ids
    FROM workspace.gcivilization.civ4_eval_questions
    WHERE category = 'beginner_strategy' LIMIT 1
""").collect()[0]
question = q_row.question
print(f"Test question: {question}")

# Package exactly like 02_model_registration will: class + code_paths + signature + example
trial_dir = tempfile.mkdtemp(prefix="civ4_rag_trial_")
mlflow.pyfunc.save_model(
    path=trial_dir,
    python_model=rag_model.Civ4RagModel(),
    code_paths=[
        "/Workspace/Repos/endoerner@gmail.com/civ4-rag/06_model_serving/rag_model.py",
        "/Workspace/Repos/endoerner@gmail.com/civ4-rag/04_generation/rag_utils.py",
    ],
    signature=sig,
    input_example=rag_model.INPUT_EXAMPLE,
    pip_requirements=["mlflow>=3.0", "databricks-sdk>=0.122.0", "pandas>=2.0"],
)
loaded = mlflow.pyfunc.load_model(trial_dir)

# Real end-to-end inference through the LOADED artifact (Vector Search + LLM)
result = loaded.predict(pd.DataFrame({"question": [question]}))
assert isinstance(result, list) and len(result) == 1
r = result[0]
assert isinstance(r["answer"], str) and len(r["answer"].strip()) > 0, "empty answer"
assert isinstance(r["references"], list) and len(r["references"]) >= 1, "no references"
ref = r["references"][0]
assert all(k in ref and ref[k] for k in ("document_id", "title", "source_url"))
assert any(s["document_id"] in q_row.expected_document_ids for s in r["references"]), \
    "retrieval did not return an expected document"
print("ANSWER (truncated):", r["answer"][:300], "...")
print(f"REFERENCES: {len(r['references'])} document(s):")
for s in r["references"]:
    print(f"  - {s['document_id']} | {s['title'][:60]}")
print("PASS  save -> load -> predict round trip from standalone artifact")

In [0]:
# Check 5 — retrieval parity with the Sprint 4 implementation (deterministic:
# embedding + ANN search are reproducible, so document-level results must match).
retrieved_direct = rag_utils.retrieve(question, top_k=rag_model.PINNED_CONFIG["top_k"])
docs_direct = {c["document_id"] for c in retrieved_direct}
docs_model = {s["document_id"] for s in r["references"]}
assert docs_direct == docs_model, f"retrieval mismatch: {docs_direct ^ docs_model}"
assert len(retrieved_direct) == 10, f"expected 10 chunks, got {len(retrieved_direct)}"
print(f"PASS  retrieval parity: {len(docs_model)} documents identical to rag_utils.retrieve(top_k=10)")

# Check 6 — out-of-domain behavior preserved (anti-hallucination system prompt).
ood = loaded.predict(pd.DataFrame({"question": ["What is the best recipe for sourdough bread?"]}))[0]
assert isinstance(ood["answer"], str) and ood["answer"].strip()
# The model must not fabricate Civ4 strategy advice for an unrelated question
print("OOD ANSWER (truncated):", ood["answer"][:300], "...")
print("PASS  out-of-domain question handled gracefully (no crash, structured response)")

In [0]:
# Summary — Sprint 6 packaging acceptance.
print("=" * 60)
print("SPRINT 6 NOTEBOOK 01: ALL PACKAGING CHECKS PASSED")
print("=" * 60)
print("""
Deliverables validated:
  - rag_model.Civ4RagModel (MLflow PyFunc, Config B pinned)
  - Signature: question -> answer + references[{document_id,title,source_url,author,category}]
  - Standalone artifact: save -> load -> predict round trip OK
  - Retrieval parity with Sprint 4 confirmed (top_k=10 documents)
  - Out-of-domain behavior preserved
  - Trial artifact at: """ + trial_dir + """ (discarded — logging happens in 02)
""")
print("READY for 02_model_registration")